# Laboratorio 7 — pipelines de regresión

Este notebook desarrolla exclusivamente los pipelines de **regresión lineal** y **Random Forest** solicitados. El objetivo es estimar `salario_mensual` con exactamente seis predictores personales y laborales.

El diseño utiliza una separación temporal reproducible:

- **Entrenamiento:** 2025T1, 2025T2 y 2025T3.
- **Validación:** 2025T4.
- **Conjunto reservado y no consultado:** 2026T1.

La selección de configuraciones se realiza únicamente con el RMSE de validación. Todos los transformadores y modelos se ajustan dentro de cada `Pipeline` usando sólo el conjunto de entrenamiento.

In [1]:
from pathlib import Path
import json
import platform
import time

import pandas as pd
import pyspark
from IPython.display import display, Markdown
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'Transform_Data'
MODELOS = OUT / 'modelos'
MODELOS.mkdir(parents=True, exist_ok=True)
PARQUET_2025 = OUT / 'personas_preparadas_2025'

assert PARQUET_2025.is_dir(), (
    'No existe Transform_Data/personas_preparadas_2025. ' 
    'Ejecute primero 01_preparacion_eneic.ipynb de principio a fin.'
)
assert pyspark.__version__.startswith('3.5.'), f'Se requiere PySpark 3.5.x; actual: {pyspark.__version__}'

spark = (SparkSession.builder
         .master('local[2]')
         .appName('L07_ENEIC_Pipelines')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.ansi.enabled', 'false')
         .config('spark.driver.memory', '2g')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')

ETIQUETA = 'salario_mensual'
NUMERICAS = ['edad', 'antiguedad', 'horas_semanales']
CATEGORICAS = ['nivel_educativo', 'categoria_ocupacional', 'dominio']
PREDICTORES = NUMERICAS + CATEGORICAS
SEMILLA = 2026

print('Python:', platform.python_version(), '| Spark:', spark.version)
print('Directorio de resultados:', OUT)

/usr/local/lib/python3.11/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/25 00:10:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/25 00:10:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Python: 3.11.16 | Spark: 3.5.1
Directorio de resultados: /opt/app/laboratorio/Transform_Data


## Diseño experimental y prevención de fuga de información

La validación es posterior al entrenamiento en el tiempo. No se usa una división aleatoria porque el propósito es comprobar si patrones aprendidos en los primeros tres trimestres se mantienen en el cuarto.

Antes de entrenar se comprueba que:

1. Estén presentes la etiqueta y los seis predictores permitidos.
2. Las variables numéricas sean finitas.
3. No existan valores nulos en las columnas del modelo.
4. Todos los registros de 2025 pertenezcan a entrenamiento o validación.

No se incorporan identificadores, FACTOR, otros ingresos, salario por hora ni la etiqueta de cluster.

In [2]:
columnas_modelo = ['periodo_archivo', ETIQUETA] + PREDICTORES
base_2025 = spark.read.parquet(str(PARQUET_2025))
faltantes_columnas = sorted(set(columnas_modelo) - set(base_2025.columns))
assert not faltantes_columnas, f'Faltan columnas requeridas: {faltantes_columnas}'
base_2025 = base_2025.select(*columnas_modelo)

controles = base_2025.agg(
    *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c + '_nulos') for c in [ETIQUETA] + PREDICTORES],
    *[F.sum(F.when(F.isnan(c) | (F.abs(F.col(c)) == float('inf')), 1).otherwise(0)).alias(c + '_no_finitos') for c in [ETIQUETA] + NUMERICAS]
).first().asDict()
problemas = {k: int(v) for k, v in controles.items() if v}
assert not problemas, f'La base preparada contiene valores inválidos: {problemas}'

entrenamiento = (base_2025
    .filter(F.col('periodo_archivo').isin('2025T1', '2025T2', '2025T3'))
    .drop('periodo_archivo').cache())
validacion = (base_2025
    .filter(F.col('periodo_archivo') == '2025T4')
    .drop('periodo_archivo').cache())

n_entrenamiento = entrenamiento.count()
n_validacion = validacion.count()
n_total = base_2025.count()
assert n_entrenamiento > 0 and n_validacion > 0
assert n_entrenamiento + n_validacion == n_total, 'Hay registros de 2025 fuera del split definido.'

display(pd.DataFrame([
    {'conjunto': 'Entrenamiento', 'periodos': '2025T1–2025T3', 'registros': n_entrenamiento, 'porcentaje': 100*n_entrenamiento/n_total},
    {'conjunto': 'Validación', 'periodos': '2025T4', 'registros': n_validacion, 'porcentaje': 100*n_validacion/n_total},
]))

,conjunto,periodos,registros,porcentaje
0,Entrenamiento,2025T1–2025T3,40361,76.116926
1,Validación,2025T4,12664,23.883074


### Análisis previo de los conjuntos

Se compara la distribución del salario entre entrenamiento y validación. Diferencias en media, mediana, dispersión o percentil 95 pueden hacer que la validación temporal sea más exigente que una división aleatoria. También se revisa si aparecen categorías en validación que no estuvieron presentes durante el ajuste.

In [3]:
def resumen_salario(nombre, df, n):
    r = df.agg(
        F.mean(ETIQUETA).alias('media'),
        F.expr(f'percentile({ETIQUETA}, 0.5)').alias('mediana'),
        F.stddev_samp(ETIQUETA).alias('desviacion_estandar'),
        F.expr(f'percentile({ETIQUETA}, 0.95)').alias('p95'),
        F.min(ETIQUETA).alias('minimo'),
        F.max(ETIQUETA).alias('maximo'),
    ).first()
    return {'conjunto': nombre, 'n': n, **r.asDict()}

resumen_conjuntos = pd.DataFrame([
    resumen_salario('Entrenamiento', entrenamiento, n_entrenamiento),
    resumen_salario('Validación', validacion, n_validacion),
])
display(resumen_conjuntos)

cobertura = []
for variable in CATEGORICAS:
    categorias_train = {r[0] for r in entrenamiento.select(variable).distinct().collect()}
    categorias_val = {r[0] for r in validacion.select(variable).distinct().collect()}
    nuevas = sorted(categorias_val - categorias_train)
    cobertura.append({
        'variable': variable,
        'categorias_entrenamiento': len(categorias_train),
        'categorias_validacion': len(categorias_val),
        'categorias_nuevas_en_validacion': ', '.join(map(str, nuevas)) if nuevas else 'Ninguna',
    })
cobertura_df = pd.DataFrame(cobertura)
display(cobertura_df)

cambio_media = 100 * (resumen_conjuntos.loc[1, 'media'] / resumen_conjuntos.loc[0, 'media'] - 1)
cambio_mediana = 100 * (resumen_conjuntos.loc[1, 'mediana'] / resumen_conjuntos.loc[0, 'mediana'] - 1)
display(Markdown(
    f'La media salarial de validación cambia **{cambio_media:+.2f}%** frente al entrenamiento y la mediana **{cambio_mediana:+.2f}%**. '
    'Estas diferencias permiten contextualizar los errores que se observarán fuera del período de ajuste.'
))

,conjunto,n,media,mediana,desviacion_estandar,p95,minimo,maximo
0,Entrenamiento,40361,3383.124997,3000.0,2906.367862,8000.0,1.0,99000.0
1,Validación,12664,3544.574937,3200.0,2885.171449,8000.0,200.0,60000.0


,variable,categorias_entrenamiento,categorias_validacion,categorias_nuevas_en_validacion
0,nivel_educativo,8,8,Ninguna
1,categoria_ocupacional,4,4,Ninguna
2,dominio,3,3,Ninguna


La media salarial de validación cambia **+4.77%** frente al entrenamiento y la mediana **+6.67%**. Estas diferencias permiten contextualizar los errores que se observarán fuera del período de ajuste.

## Modelo de referencia y métricas

El modelo de referencia predice la media salarial del entrenamiento para todos los registros. Es deliberadamente simple, pero permite comprobar si los modelos aportan información útil.

Se reportan tres métricas:

- **MAE:** error absoluto medio en quetzales; es fácil de interpretar y menos sensible a errores extremos que RMSE.
- **RMSE:** penaliza con mayor fuerza los errores grandes y es el criterio obligatorio de selección.
- **R²:** proporción de variabilidad explicada respecto a una referencia constante. Puede ser negativo en validación si el modelo generaliza peor que esa referencia.

Además se calcula el RMSE de entrenamiento. La diferencia porcentual entre RMSE de validación y entrenamiento se usa como señal descriptiva de posible sobreajuste.

In [4]:
def calcular_metricas(predicciones):
    evaluable = predicciones.select(ETIQUETA, 'prediction').cache()
    evaluable.count()
    resultado = {
        'mae': float(RegressionEvaluator(labelCol=ETIQUETA, predictionCol='prediction', metricName='mae').evaluate(evaluable)),
        'rmse': float(RegressionEvaluator(labelCol=ETIQUETA, predictionCol='prediction', metricName='rmse').evaluate(evaluable)),
        'r2': float(RegressionEvaluator(labelCol=ETIQUETA, predictionCol='prediction', metricName='r2').evaluate(evaluable)),
    }
    evaluable.unpersist()
    return resultado

media_entrenamiento = float(entrenamiento.agg(F.mean(ETIQUETA)).first()[0])
pred_baseline_train = entrenamiento.select(ETIQUETA).withColumn('prediction', F.lit(media_entrenamiento))
pred_baseline_val = validacion.select(ETIQUETA).withColumn('prediction', F.lit(media_entrenamiento))
metricas_baseline_train = calcular_metricas(pred_baseline_train)
metricas_baseline_val = calcular_metricas(pred_baseline_val)

fila_baseline = {
    'algoritmo': 'Baseline', 'configuracion': 'Media del entrenamiento',
    'mae_entrenamiento': metricas_baseline_train['mae'],
    'rmse_entrenamiento': metricas_baseline_train['rmse'],
    'r2_entrenamiento': metricas_baseline_train['r2'],
    'mae_validacion': metricas_baseline_val['mae'],
    'rmse_validacion': metricas_baseline_val['rmse'],
    'r2_validacion': metricas_baseline_val['r2'],
}
display(pd.DataFrame([fila_baseline]))
display(Markdown(
    f'El baseline predice **Q{media_entrenamiento:,.2f}**. En validación obtiene MAE **Q{metricas_baseline_val["mae"]:,.2f}**, '
    f'RMSE **Q{metricas_baseline_val["rmse"]:,.2f}** y R² **{metricas_baseline_val["r2"]:.3f}**.'
))

,algoritmo,configuracion,mae_entrenamiento,rmse_entrenamiento,r2_entrenamiento,mae_validacion,rmse_validacion,r2_validacion
0,Baseline,Media del entrenamiento,1696.306227,2906.331857,3.097522e-14,1672.502606,2889.571432,-0.003132


El baseline predice **Q3,383.12**. En validación obtiene MAE **Q1,672.50**, RMSE **Q2,889.57** y R² **-0.003**.

## Preprocesamiento común dentro de los pipelines

Las tres variables categóricas se procesan con un `StringIndexer` independiente y después con `OneHotEncoder`. Las tres variables numéricas se incorporan directamente al `VectorAssembler`.

`handleInvalid="keep"` evita errores si validación contiene una categoría que no apareció en entrenamiento. Esto no significa ajustar el índice con validación: el valor nuevo se dirige a una posición especial aprendida por la configuración del transformador.

La función devuelve estimadores nuevos en cada llamada. De este modo cada configuración se ajusta desde cero únicamente con entrenamiento.

In [5]:
def etapas_preprocesamiento(output_features='features'):
    columnas_indices = [f'{c}_indice' for c in CATEGORICAS]
    columnas_ohe = [f'{c}_ohe' for c in CATEGORICAS]
    indexadores = [
        StringIndexer(inputCol=entrada, outputCol=salida, handleInvalid='keep', stringOrderType='alphabetAsc')
        for entrada, salida in zip(CATEGORICAS, columnas_indices)
    ]
    codificador = OneHotEncoder(
        inputCols=columnas_indices, outputCols=columnas_ohe, handleInvalid='keep', dropLast=True
    )
    ensamblador = VectorAssembler(
        inputCols=NUMERICAS + columnas_ohe, outputCol=output_features, handleInvalid='error'
    )
    return indexadores + [codificador, ensamblador]

print('Variables numéricas:', NUMERICAS)
print('Variables categóricas:', CATEGORICAS)
print('Orden de entrada al VectorAssembler:', NUMERICAS + [f'{c}_ohe' for c in CATEGORICAS])

Variables numéricas: ['edad', 'antiguedad', 'horas_semanales']
Variables categóricas: ['nivel_educativo', 'categoria_ocupacional', 'dominio']
Orden de entrada al VectorAssembler: ['edad', 'antiguedad', 'horas_semanales', 'nivel_educativo_ohe', 'categoria_ocupacional_ohe', 'dominio_ohe']


## 5. Pipeline de regresión lineal

El pipeline contiene `StringIndexer`, `OneHotEncoder`, `VectorAssembler` y `LinearRegression`. Se usa `standardization=True`, que estandariza internamente los predictores durante la optimización; no se agrega `StandardScaler` porque aplicar ambos mecanismos sería redundante.

La búsqueda incluye una referencia sin regularización y configuraciones Ridge y Elastic Net. Los valores más amplios permiten verificar si una penalización apreciable reduce el error o si introduce subajuste. `elasticNetParam=0` corresponde a Ridge y `elasticNetParam=0.5` mezcla penalizaciones L1 y L2.

In [6]:
CONFIGURACIONES_LR = [
    {'nombre': 'sin_regularizacion', 'regParam': 0.0, 'elasticNetParam': 0.0, 'tipo': 'Sin penalización'},
    {'nombre': 'ridge_0.10', 'regParam': 0.10, 'elasticNetParam': 0.0, 'tipo': 'Ridge'},
    {'nombre': 'ridge_1.00', 'regParam': 1.00, 'elasticNetParam': 0.0, 'tipo': 'Ridge'},
    {'nombre': 'ridge_10.00', 'regParam': 10.00, 'elasticNetParam': 0.0, 'tipo': 'Ridge'},
    {'nombre': 'ridge_50.00', 'regParam': 50.00, 'elasticNetParam': 0.0, 'tipo': 'Ridge'},
    {'nombre': 'elastic_net_1.00', 'regParam': 1.00, 'elasticNetParam': 0.5, 'tipo': 'Elastic Net'},
    {'nombre': 'elastic_net_10.00', 'regParam': 10.00, 'elasticNetParam': 0.5, 'tipo': 'Elastic Net'},
]
display(pd.DataFrame(CONFIGURACIONES_LR))

,nombre,regParam,elasticNetParam,tipo
0,sin_regularizacion,0.0,0.0,Sin penalización
1,ridge_0.10,0.1,0.0,Ridge
2,ridge_1.00,1.0,0.0,Ridge
3,ridge_10.00,10.0,0.0,Ridge
4,ridge_50.00,50.0,0.0,Ridge
5,elastic_net_1.00,1.0,0.5,Elastic Net
6,elastic_net_10.00,10.0,0.5,Elastic Net


### Ajuste y evaluación de la regresión lineal

Cada fila de la tabla anterior produce un pipeline completo e independiente. Se registran métricas tanto en entrenamiento como en validación, pero la selección utiliza únicamente `rmse_validacion`. También se calcula la mejora porcentual frente al baseline y la brecha de generalización.

In [7]:
resultados_lr, mejor_modelo_lr, mejor_fila_lr = [], None, None
for configuracion in CONFIGURACIONES_LR:
    estimador = LinearRegression(
        featuresCol='features', labelCol=ETIQUETA, predictionCol='prediction',
        standardization=True, fitIntercept=True, maxIter=100, tol=1e-6,
        regParam=configuracion['regParam'], elasticNetParam=configuracion['elasticNetParam'],
    )
    pipeline = Pipeline(stages=etapas_preprocesamiento() + [estimador])
    inicio = time.perf_counter()
    modelo = pipeline.fit(entrenamiento)
    segundos = time.perf_counter() - inicio
    metricas_train = calcular_metricas(modelo.transform(entrenamiento))
    metricas_val = calcular_metricas(modelo.transform(validacion))
    fila = {
        'algoritmo': 'Regresión lineal', 'configuracion': configuracion['nombre'],
        'tipo': configuracion['tipo'], 'regParam': configuracion['regParam'],
        'elasticNetParam': configuracion['elasticNetParam'], 'segundos_ajuste': segundos,
        'mae_entrenamiento': metricas_train['mae'], 'rmse_entrenamiento': metricas_train['rmse'],
        'r2_entrenamiento': metricas_train['r2'], 'mae_validacion': metricas_val['mae'],
        'rmse_validacion': metricas_val['rmse'], 'r2_validacion': metricas_val['r2'],
        'mejora_rmse_vs_baseline_pct': 100 * (metricas_baseline_val['rmse'] - metricas_val['rmse']) / metricas_baseline_val['rmse'],
        'brecha_rmse_pct': 100 * (metricas_val['rmse'] / metricas_train['rmse'] - 1),
    }
    resultados_lr.append(fila)
    if mejor_fila_lr is None or fila['rmse_validacion'] < mejor_fila_lr['rmse_validacion']:
        mejor_fila_lr, mejor_modelo_lr = fila, modelo
    print(configuracion['nombre'], '| RMSE validación:', f"{metricas_val['rmse']:.2f}")

resultados_lr_df = pd.DataFrame(resultados_lr).sort_values('rmse_validacion').reset_index(drop=True)
display(resultados_lr_df)
ruta_lr = MODELOS / 'validacion_regresion_lineal'
mejor_modelo_lr.write().overwrite().save(str(ruta_lr))
print('Etapas del mejor pipeline:', [type(etapa).__name__ for etapa in mejor_modelo_lr.stages])
print('Modelo guardado en:', ruta_lr)

26/09/25 00:11:15 WARN Instrumentation: [026bf11c] regParam is zero, which might cause numerical instability and overfitting.
26/09/25 00:11:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/09/25 00:11:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/09/25 00:11:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/09/25 00:11:16 WARN Instrumentation: [026bf11c] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.


sin_regularizacion | RMSE validación: 2186.42
ridge_0.10 | RMSE validación: 2186.42
ridge_1.00 | RMSE validación: 2186.43
ridge_10.00 | RMSE validación: 2186.55
ridge_50.00 | RMSE validación: 2187.15
elastic_net_1.00 | RMSE validación: 2186.47
elastic_net_10.00 | RMSE validación: 2186.89


,algoritmo,configuracion,tipo,regParam,elasticNetParam,segundos_ajuste,mae_entrenamiento,rmse_entrenamiento,r2_entrenamiento,mae_validacion,rmse_validacion,r2_validacion,mejora_rmse_vs_baseline_pct,brecha_rmse_pct
0,Regresión lineal,sin_regularizacion,Sin penalización,0.0,0.0,4.822620,1237.790468,2245.292688,0.403163,1210.137480,2186.419579,0.425675,24.334123,-2.622068
1,Regresión lineal,ridge_0.10,Ridge,0.1,0.0,0.992114,1237.786618,2245.292688,0.403163,1210.132737,2186.420865,0.425674,24.334078,-2.622011
2,Regresión lineal,ridge_1.00,Ridge,1.0,0.0,0.664696,1237.752143,2245.292731,0.403163,1210.090188,2186.432421,0.425668,24.333678,-2.621498
3,Regresión lineal,elastic_net_1.00,Elastic Net,1.0,0.5,0.716032,1237.556008,2245.293680,0.403163,1209.852865,2186.469753,0.425648,24.332386,-2.619877
4,Regresión lineal,ridge_10.00,Ridge,10.0,0.0,0.777272,1237.411638,2245.296931,0.403161,1209.670472,2186.551423,0.425605,24.329560,-2.616380
5,Regresión lineal,elastic_net_10.00,Elastic Net,10.0,0.5,0.747883,1235.942599,2245.362018,0.403126,1207.806090,2186.891285,0.425427,24.317798,-2.604067
6,Regresión lineal,ridge_50.00,Ridge,50.0,0.0,0.892971,1235.976702,2245.395880,0.403108,1207.909570,2187.153440,0.425289,24.308726,-2.593861


Etapas del mejor pipeline: ['StringIndexerModel', 'StringIndexerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'LinearRegressionModel']
Modelo guardado en: /opt/app/laboratorio/Transform_Data/modelos/validacion_regresion_lineal


### Análisis de regularización

El análisis siguiente compara la configuración ganadora con la versión sin regularización. Una diferencia mínima indicaría que las penalizaciones no cambian materialmente la capacidad de generalización con estas variables; una penalización excesiva puede aumentar el error al reducir demasiado los coeficientes.

In [8]:
fila_sin_reg = next(f for f in resultados_lr if f['configuracion'] == 'sin_regularizacion')
cambio_regularizacion = 100 * (fila_sin_reg['rmse_validacion'] - mejor_fila_lr['rmse_validacion']) / fila_sin_reg['rmse_validacion']
if abs(cambio_regularizacion) < 0.10:
    lectura_regularizacion = 'La diferencia es menor a 0.10%, por lo que la regularización no cambia materialmente el desempeño.'
elif cambio_regularizacion > 0:
    lectura_regularizacion = 'La penalización reduce el RMSE y mejora la generalización respecto al modelo sin regularización.'
else:
    lectura_regularizacion = 'La configuración sin regularización conserva el menor RMSE; las penalizaciones probadas introducen subajuste.'
display(Markdown(f'''
La mejor regresión lineal es **{mejor_fila_lr['configuracion']}**: MAE de validación **Q{mejor_fila_lr['mae_validacion']:,.2f}**, RMSE **Q{mejor_fila_lr['rmse_validacion']:,.2f}** y R² **{mejor_fila_lr['r2_validacion']:.3f}**. Mejora el RMSE del baseline en **{mejor_fila_lr['mejora_rmse_vs_baseline_pct']:.2f}%**.

Frente a la versión sin regularización, el cambio de RMSE es **{cambio_regularizacion:+.3f}%**. {lectura_regularizacion} La brecha entre RMSE de entrenamiento y validación es **{mejor_fila_lr['brecha_rmse_pct']:+.2f}%**.
'''))


La mejor regresión lineal es **sin_regularizacion**: MAE de validación **Q1,210.14**, RMSE **Q2,186.42** y R² **0.426**. Mejora el RMSE del baseline en **24.33%**.

Frente a la versión sin regularización, el cambio de RMSE es **+0.000%**. La diferencia es menor a 0.10%, por lo que la regularización no cambia materialmente el desempeño. La brecha entre RMSE de entrenamiento y validación es **-2.62%**.


## 6. Pipeline de Random Forest

Random Forest utiliza los mismos `StringIndexer`, `OneHotEncoder` y `VectorAssembler`, pero no requiere estandarización porque las divisiones de los árboles no dependen de la escala.

Las pruebas permiten estudiar dos efectos:

- Aumentar `maxDepth` incrementa la capacidad para representar interacciones, pero también el riesgo de sobreajuste.
- Aumentar `numTrees` suele estabilizar el promedio del bosque a cambio de mayor tiempo de cómputo.

Todas las configuraciones usan la misma semilla, tasa de muestreo y reglas mínimas por nodo para que la comparación sea reproducible.

In [9]:
CONFIGURACIONES_RF = [
    {'nombre': 'rf_50_arboles_d8', 'numTrees': 50, 'maxDepth': 8},
    {'nombre': 'rf_100_arboles_d10', 'numTrees': 100, 'maxDepth': 10},
    {'nombre': 'rf_100_arboles_d12', 'numTrees': 100, 'maxDepth': 12},
    {'nombre': 'rf_200_arboles_d12', 'numTrees': 200, 'maxDepth': 12},
    {'nombre': 'rf_100_arboles_d14', 'numTrees': 100, 'maxDepth': 14},
]
display(pd.DataFrame(CONFIGURACIONES_RF).assign(semilla=SEMILLA, subsamplingRate=0.8))

,nombre,numTrees,maxDepth,semilla,subsamplingRate
0,rf_50_arboles_d8,50,8,2026,0.8
1,rf_100_arboles_d10,100,10,2026,0.8
2,rf_100_arboles_d12,100,12,2026,0.8
3,rf_200_arboles_d12,200,12,2026,0.8
4,rf_100_arboles_d14,100,14,2026,0.8


### Ajuste y evaluación de Random Forest

Se reutilizan exactamente las mismas filas de entrenamiento y validación que en regresión lineal. Para cada bosque se registran tiempo, métricas, mejora frente al baseline y brecha entre entrenamiento y validación. El modelo guardado es el de menor RMSE de validación.

In [10]:
resultados_rf, mejor_modelo_rf, mejor_fila_rf = [], None, None
for configuracion in CONFIGURACIONES_RF:
    estimador = RandomForestRegressor(
        featuresCol='features', labelCol=ETIQUETA, predictionCol='prediction',
        numTrees=configuracion['numTrees'], maxDepth=configuracion['maxDepth'], seed=SEMILLA,
        subsamplingRate=0.8, featureSubsetStrategy='auto', maxBins=32, minInstancesPerNode=2,
    )
    pipeline = Pipeline(stages=etapas_preprocesamiento() + [estimador])
    inicio = time.perf_counter()
    modelo = pipeline.fit(entrenamiento)
    segundos = time.perf_counter() - inicio
    metricas_train = calcular_metricas(modelo.transform(entrenamiento))
    metricas_val = calcular_metricas(modelo.transform(validacion))
    fila = {
        'algoritmo': 'Random Forest', 'configuracion': configuracion['nombre'],
        'numTrees': configuracion['numTrees'], 'maxDepth': configuracion['maxDepth'],
        'semilla': SEMILLA, 'segundos_ajuste': segundos,
        'mae_entrenamiento': metricas_train['mae'], 'rmse_entrenamiento': metricas_train['rmse'],
        'r2_entrenamiento': metricas_train['r2'], 'mae_validacion': metricas_val['mae'],
        'rmse_validacion': metricas_val['rmse'], 'r2_validacion': metricas_val['r2'],
        'mejora_rmse_vs_baseline_pct': 100 * (metricas_baseline_val['rmse'] - metricas_val['rmse']) / metricas_baseline_val['rmse'],
        'brecha_rmse_pct': 100 * (metricas_val['rmse'] / metricas_train['rmse'] - 1),
    }
    resultados_rf.append(fila)
    if mejor_fila_rf is None or fila['rmse_validacion'] < mejor_fila_rf['rmse_validacion']:
        mejor_fila_rf, mejor_modelo_rf = fila, modelo
    print(configuracion['nombre'], '| RMSE validación:', f"{metricas_val['rmse']:.2f}")

resultados_rf_df = pd.DataFrame(resultados_rf).sort_values('rmse_validacion').reset_index(drop=True)
display(resultados_rf_df)
ruta_rf = MODELOS / 'validacion_random_forest'
mejor_modelo_rf.write().overwrite().save(str(ruta_rf))
print('Etapas del mejor pipeline:', [type(etapa).__name__ for etapa in mejor_modelo_rf.stages])
print('Modelo guardado en:', ruta_rf)

26/09/25 00:11:44 WARN DAGScheduler: Broadcasting large task binary with size 1013.7 KiB
26/09/25 00:11:45 WARN DAGScheduler: Broadcasting large task binary with size 1714.8 KiB
                                                                                

rf_50_arboles_d8 | RMSE validación: 2047.63


26/09/25 00:11:53 WARN DAGScheduler: Broadcasting large task binary with size 1070.6 KiB
26/09/25 00:11:55 WARN DAGScheduler: Broadcasting large task binary with size 1861.5 KiB
26/09/25 00:11:57 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/09/25 00:12:00 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/25 00:12:04 WARN DAGScheduler: Broadcasting large task binary with size 1174.5 KiB
26/09/25 00:12:05 WARN DAGScheduler: Broadcasting large task binary with size 8.0 MiB
26/09/25 00:12:10 WARN DAGScheduler: Broadcasting large task binary with size 1731.7 KiB
                                                                                

rf_100_arboles_d10 | RMSE validación: 2005.72


26/09/25 00:12:19 WARN DAGScheduler: Broadcasting large task binary with size 1070.6 KiB
26/09/25 00:12:20 WARN DAGScheduler: Broadcasting large task binary with size 1861.5 KiB
26/09/25 00:12:22 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/09/25 00:12:25 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/25 00:12:28 WARN DAGScheduler: Broadcasting large task binary with size 1174.5 KiB
26/09/25 00:12:29 WARN DAGScheduler: Broadcasting large task binary with size 8.0 MiB
26/09/25 00:12:32 WARN DAGScheduler: Broadcasting large task binary with size 1731.7 KiB
26/09/25 00:12:33 WARN DAGScheduler: Broadcasting large task binary with size 12.0 MiB
26/09/25 00:12:39 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/09/25 00:12:41 WARN DAGScheduler: Broadcasting large task binary with size 17.1 MiB
26/09/25 00:12:46 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
                                        

...

26/09/25 00:14:27 WARN DAGScheduler: Broadcasting large task binary with size 1070.6 KiB
26/09/25 00:14:31 WARN DAGScheduler: Broadcasting large task binary with size 1861.5 KiB
26/09/25 00:14:33 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/09/25 00:14:37 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/09/25 00:14:40 WARN DAGScheduler: Broadcasting large task binary with size 1174.5 KiB
26/09/25 00:14:41 WARN DAGScheduler: Broadcasting large task binary with size 8.0 MiB
26/09/25 00:14:46 WARN DAGScheduler: Broadcasting large task binary with size 1731.7 KiB
26/09/25 00:14:47 WARN DAGScheduler: Broadcasting large task binary with size 12.0 MiB
26/09/25 00:14:53 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/09/25 00:14:56 WARN DAGScheduler: Broadcasting large task binary with size 17.1 MiB
26/09/25 00:15:01 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/09/25 00:15:04 WARN DAGScheduler: Bro

rf_100_arboles_d14 | RMSE validación: 1959.45


,algoritmo,configuracion,numTrees,maxDepth,semilla,segundos_ajuste,mae_entrenamiento,rmse_entrenamiento,r2_entrenamiento,mae_validacion,rmse_validacion,r2_validacion,mejora_rmse_vs_baseline_pct,brecha_rmse_pct
0,Random Forest,rf_100_arboles_d14,100,14,2026,60.382670,1022.946934,1877.791846,0.582550,1057.819059,1959.452972,0.538724,32.188803,4.348785
1,Random Forest,rf_200_arboles_d12,200,12,2026,74.434797,1046.539754,1916.292677,0.565256,1066.254189,1972.970006,0.532338,31.721016,2.957655
2,Random Forest,rf_100_arboles_d12,100,12,2026,32.223671,1048.684017,1917.018992,0.564927,1068.271805,1978.544272,0.529692,31.528107,3.209425
3,Random Forest,rf_100_arboles_d10,100,10,2026,22.206859,1077.851869,1977.433848,0.537072,1080.465592,2005.721934,0.516683,30.587564,1.430545
4,Random Forest,rf_50_arboles_d8,50,8,2026,6.044954,1109.855034,2058.033534,0.498565,1099.425983,2047.629015,0.496275,29.137276,-0.505556


26/09/25 00:15:42 WARN TaskSetManager: Stage 522 contains a task of very large size (9836 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Etapas del mejor pipeline: ['StringIndexerModel', 'StringIndexerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'RandomForestRegressionModel']
Modelo guardado en: /opt/app/laboratorio/Transform_Data/modelos/validacion_random_forest


### Análisis de árboles profundidad y sobreajuste

La mejor configuración se interpreta junto con su brecha de generalización. Una reducción continua del error de entrenamiento acompañada por un empeoramiento en validación indicaría que aumentar la profundidad ya no ayuda fuera del período de ajuste.

In [11]:
fila_rf_simple = next(f for f in resultados_rf if f['configuracion'] == 'rf_50_arboles_d8')
mejora_complejidad = 100 * (fila_rf_simple['rmse_validacion'] - mejor_fila_rf['rmse_validacion']) / fila_rf_simple['rmse_validacion']
display(Markdown(f'''
El mejor bosque es **{mejor_fila_rf['configuracion']}**: MAE de validación **Q{mejor_fila_rf['mae_validacion']:,.2f}**, RMSE **Q{mejor_fila_rf['rmse_validacion']:,.2f}** y R² **{mejor_fila_rf['r2_validacion']:.3f}**. Mejora el RMSE del baseline en **{mejor_fila_rf['mejora_rmse_vs_baseline_pct']:.2f}%**.

Respecto al bosque inicial de 50 árboles y profundidad 8, la configuración seleccionada cambia el RMSE en **{mejora_complejidad:+.2f}%** de mejora. Su brecha entre entrenamiento y validación es **{mejor_fila_rf['brecha_rmse_pct']:+.2f}%**; debe leerse junto con la tabla completa para determinar si mayor profundidad está produciendo ganancias reales o sobreajuste.
'''))


El mejor bosque es **rf_100_arboles_d14**: MAE de validación **Q1,057.82**, RMSE **Q1,959.45** y R² **0.539**. Mejora el RMSE del baseline en **32.19%**.

Respecto al bosque inicial de 50 árboles y profundidad 8, la configuración seleccionada cambia el RMSE en **+4.31%** de mejora. Su brecha entre entrenamiento y validación es **+4.35%**; debe leerse junto con la tabla completa para determinar si mayor profundidad está produciendo ganancias reales o sobreajuste.


## Comparación final de validación

Se comparan el baseline y las configuraciones ganadoras sobre exactamente los mismos registros de 2025T4. La tabla se ordena por RMSE porque ése fue el criterio de selección definido antes de observar los resultados.

In [12]:
comparacion = pd.DataFrame([fila_baseline, mejor_fila_lr, mejor_fila_rf])
columnas_comparacion = [
    'algoritmo', 'configuracion', 'mae_validacion', 'rmse_validacion', 'r2_validacion',
    'rmse_entrenamiento', 'brecha_rmse_pct', 'mejora_rmse_vs_baseline_pct'
]
for columna in columnas_comparacion:
    if columna not in comparacion.columns:
        comparacion[columna] = 0.0
comparacion = comparacion[columnas_comparacion].sort_values('rmse_validacion').reset_index(drop=True)
display(comparacion)

comparacion.to_csv(OUT / 'comparacion_modelos_validacion.csv', index=False)
resultados_lr_df.to_csv(OUT / 'configuraciones_regresion_lineal.csv', index=False)
resultados_rf_df.to_csv(OUT / 'configuraciones_random_forest.csv', index=False)

seleccion = {
    'criterio': 'menor RMSE en 2025T4',
    'entrenamiento': ['2025T1', '2025T2', '2025T3'], 'validacion': '2025T4',
    'conjunto_no_consultado': '2026T1', 'semilla_random_forest': SEMILLA,
    'regresion_lineal': {k: v for k, v in mejor_fila_lr.items() if k != 'segundos_ajuste'},
    'random_forest': {k: v for k, v in mejor_fila_rf.items() if k != 'segundos_ajuste'},
    'baseline': fila_baseline,
}
(OUT / 'seleccion_modelos_validacion.json').write_text(
    json.dumps(seleccion, indent=2, ensure_ascii=False), encoding='utf-8'
)

,algoritmo,configuracion,mae_validacion,rmse_validacion,r2_validacion,rmse_entrenamiento,brecha_rmse_pct,mejora_rmse_vs_baseline_pct
0,Random Forest,rf_100_arboles_d14,1057.819059,1959.452972,0.538724,1877.791846,4.348785,32.188803
1,Regresión lineal,sin_regularizacion,1210.137480,2186.419579,0.425675,2245.292688,-2.622068,24.334123
2,Baseline,Media del entrenamiento,1672.502606,2889.571432,-0.003132,2906.331857,NaN,NaN


1638

## Conclusiones y descubrimientos


In [13]:
ganador = comparacion.iloc[0]
ventaja_rf_sobre_lr = 100 * (mejor_fila_lr['rmse_validacion'] - mejor_fila_rf['rmse_validacion']) / mejor_fila_lr['rmse_validacion']
if mejor_fila_rf['rmse_validacion'] < mejor_fila_lr['rmse_validacion']:
    explicacion_algoritmos = (
        'Random Forest obtiene el menor error. Su ventaja puede explicarse porque representa interacciones y relaciones no lineales '
        'entre edad, antigüedad, horas y categorías sin imponer una única relación aditiva.'
    )
else:
    explicacion_algoritmos = (
        'La regresión lineal obtiene el menor error. Esto sugiere que, para estas variables y este período, una estructura aditiva '
        'generaliza mejor que la complejidad adicional del bosque.'
    )

display(Markdown(f'''
1. **Modelo de referencia.** El baseline alcanza RMSE **Q{metricas_baseline_val['rmse']:,.2f}** y R² **{metricas_baseline_val['r2']:.3f}** en validación.
2. **Regresión lineal.** La configuración seleccionada es **{mejor_fila_lr['configuracion']}**, con RMSE **Q{mejor_fila_lr['rmse_validacion']:,.2f}**, MAE **Q{mejor_fila_lr['mae_validacion']:,.2f}** y R² **{mejor_fila_lr['r2_validacion']:.3f}**. Su mejora de RMSE frente al baseline es **{mejor_fila_lr['mejora_rmse_vs_baseline_pct']:.2f}%**.
3. **Random Forest.** La configuración seleccionada es **{mejor_fila_rf['configuracion']}**, con RMSE **Q{mejor_fila_rf['rmse_validacion']:,.2f}**, MAE **Q{mejor_fila_rf['mae_validacion']:,.2f}** y R² **{mejor_fila_rf['r2_validacion']:.3f}**. Su mejora frente al baseline es **{mejor_fila_rf['mejora_rmse_vs_baseline_pct']:.2f}%**.
4. **Comparación de algoritmos.** El cambio de RMSE de Random Forest respecto a regresión lineal es **{ventaja_rf_sobre_lr:+.2f}%** de mejora. {explicacion_algoritmos}
5. **Generalización.** Las brechas RMSE entrenamiento-validación son **{mejor_fila_lr['brecha_rmse_pct']:+.2f}%** para regresión lineal y **{mejor_fila_rf['brecha_rmse_pct']:+.2f}%** para Random Forest. Una brecha mayor en el bosque es coherente con su mayor flexibilidad y debe considerarse al interpretar su ventaja.
6. **Selección reproducible.** Las configuraciones se compararon sobre el mismo trimestre, con transformadores ajustados sólo en entrenamiento, semilla fija para el bosque y selección previa por RMSE.
'''))

entrenamiento.unpersist()
validacion.unpersist()
print('Pipelines 5 y 6 terminados. Resultados y modelos guardados en:', OUT)


1. **Modelo de referencia.** El baseline alcanza RMSE **Q2,889.57** y R² **-0.003** en validación.
2. **Regresión lineal.** La configuración seleccionada es **sin_regularizacion**, con RMSE **Q2,186.42**, MAE **Q1,210.14** y R² **0.426**. Su mejora de RMSE frente al baseline es **24.33%**.
3. **Random Forest.** La configuración seleccionada es **rf_100_arboles_d14**, con RMSE **Q1,959.45**, MAE **Q1,057.82** y R² **0.539**. Su mejora frente al baseline es **32.19%**.
4. **Comparación de algoritmos.** El cambio de RMSE de Random Forest respecto a regresión lineal es **+10.38%** de mejora. Random Forest obtiene el menor error. Su ventaja puede explicarse porque representa interacciones y relaciones no lineales entre edad, antigüedad, horas y categorías sin imponer una única relación aditiva.
5. **Generalización.** Las brechas RMSE entrenamiento-validación son **-2.62%** para regresión lineal y **+4.35%** para Random Forest. Una brecha mayor en el bosque es coherente con su mayor flexibilidad y debe considerarse al interpretar su ventaja.
6. **Selección reproducible.** Las configuraciones se compararon sobre el mismo trimestre, con transformadores ajustados sólo en entrenamiento, semilla fija para el bosque y selección previa por RMSE.


Pipelines 5 y 6 terminados. Resultados y modelos guardados en: /opt/app/laboratorio/Transform_Data


## Análisis

El modelo de referencia establece el nivel mínimo de desempeño que deberían superar los algoritmos. Al predecir siempre la media salarial del entrenamiento, ignora todas las características personales y laborales. Por ello, una disminución de MAE y RMSE y un aumento de R² frente a este baseline indican que edad, antigüedad, horas semanales, nivel educativo, categoría ocupacional y dominio contienen información útil para estimar el salario mensual. La comparación sigue siendo predictiva y descriptiva: no demuestra que alguna de estas variables cause cambios salariales.

La regresión lineal proporciona una primera aproximación estructurada. Su principal fortaleza es resumir el salario como una combinación aditiva de variables numéricas y categorías codificadas. La prueba de diferentes valores de `regParam` permite observar si reducir la magnitud de los coeficientes mejora la generalización. Si las configuraciones regularizadas producen métricas casi idénticas al modelo sin penalización, la regularización aporta poca mejora con estos predictores; si una penalización fuerte incrementa el RMSE, existe evidencia de subajuste. La selección debe basarse en el menor RMSE de validación, no en cuál configuración parece más compleja.

Random Forest amplía la capacidad del análisis porque puede representar relaciones no lineales e interacciones. Por ejemplo, el efecto asociado con la educación puede variar entre categorías ocupacionales o dominios sin que esas interacciones deban escribirse manualmente. Aumentar la profundidad permite representar reglas más específicas, mientras que aumentar el número de árboles estabiliza el promedio del bosque. Sin embargo, una reducción del error de entrenamiento acompañada por una brecha creciente frente a validación es una señal de posible sobreajuste. Por esta razón, las métricas de entrenamiento se usan como diagnóstico, pero la elección final continúa dependiendo del RMSE de validación.

En conjunto, el mejor algoritmo es el que obtiene el menor RMSE sobre exactamente los mismos registros de 2025T4, siempre que su ventaja se interprete junto con MAE, R² y la brecha de generalización. MAE expresa el error típico en quetzales sin penalizar excesivamente los casos extremos; RMSE revela si existen errores grandes importantes; y R² muestra cuánto mejora el modelo respecto a una predicción constante. Revisar las tres métricas evita concluir que un modelo es superior por un único indicador aislado.